In [13]:
import pandas as pd
import re

df = pd.read_csv(r"C:\Users\pesak\Downloads\archive (1)\IMDB Dataset.csv")
print(df.head())
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

def clean_text(text):
    text = str(text).lower()                 
    text = re.sub(r"<.*?>", " ", text) 
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df["review"].apply(clean_text)


df = df[["clean_review", "label"]]
print("\nCleaned data:")
print(df.head())
print("\nLabel distribution:")
print(df["label"].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Shape: (50000, 2)

Missing values:
review       0
sentiment    0
dtype: int64

Cleaned data:
                                        clean_review  label
0  one of the other reviewers has mentioned that ...      1
1  a wonderful little production the filming tech...      1
2  i thought this was a wonderful way to spend ti...      1
3  basically there s a family where a little boy ...      0
4  petter mattei s love in the time of money is a...      1

Label distribution:
label
1    25000
0    25000
Name: count, dtype: int64


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["clean_review"]
y = df["label"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

vectorizer = TfidfVectorizer(
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


print("TFIDF train shape:", X_train_tfidf.shape)
print("TFIDF test shape:", X_test_tfidf.shape)

Train size: 40000
Test size: 10000
TFIDF train shape: (40000, 5000)
TFIDF test shape: (10000, 5000)


In [15]:
import torch
import numpy as np

X_train_array = X_train_tfidf.toarray()
X_test_array = X_test_tfidf.toarray()

X_train_tensor = torch.tensor(X_train_array, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_array, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

print("X_train_tensor shape:", X_train_tensor.shape)
print("X_test_tensor shape:", X_test_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)

X_train_tensor shape: torch.Size([40000, 5000])
X_test_tensor shape: torch.Size([10000, 5000])
y_train_tensor shape: torch.Size([40000])
y_test_tensor shape: torch.Size([10000])


In [16]:
import torch.nn as nn

class SentimentNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.output = nn.Linear(64, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))
        x = self.dropout(x)

        x = self.output(x)
        return x

model = SentimentNet(input_size=5000)
print(model)

SentimentNet(
  (fc1): Linear(in_features=5000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)


In [17]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()

    predictions = model(X_train_tensor).squeeze()
    loss = loss_function(predictions, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/10, Loss: 0.6931
Epoch 2/10, Loss: 0.6922
Epoch 3/10, Loss: 0.6909
Epoch 4/10, Loss: 0.6892
Epoch 5/10, Loss: 0.6870
Epoch 6/10, Loss: 0.6847
Epoch 7/10, Loss: 0.6818
Epoch 8/10, Loss: 0.6786
Epoch 9/10, Loss: 0.6749
Epoch 10/10, Loss: 0.6709


In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()

with torch.no_grad():
    test_logits = model(X_test_tensor).squeeze()
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs > 0.5).float()

y_true = y_test_tensor.numpy()
y_pred = test_preds.numpy()

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 Score :", f1_score(y_true, y_pred))


Accuracy : 0.8229
Precision: 0.8388247639034627
Recall   : 0.7994
F1 Score : 0.8186379928315413
